In [1]:
pip install fastapi uvicorn prometheus-client

In [7]:
%%writefile app.py
from time import time
from fastapi import FastAPI, Request
from prometheus_client import Counter, Histogram, generate_latest, CONTENT_TYPE_LATEST
from starlette.responses import Response

app = FastAPI(title="Prometheus Monitored Service")

REQUEST_COUNT = Counter(
    "app_requests_total",
    "Total number of HTTP requests processed",
    ["method", "endpoint", "http_status"],
)

REQUEST_LATENCY = Histogram(
    "app_request_latency_seconds",
    "Time spent processing HTTP request in seconds",
    ["endpoint"],
)

@app.middleware("http")
async def monitor_requests(request: Request, call_next):
    start_time = time()
    response = await call_next(request)

    process_time = time() - start_time
    endpoint = request.url.path
    method = request.method
    status_code = str(response.status_code)

    REQUEST_COUNT.labels(
        method=method, endpoint=endpoint, http_status=status_code
    ).inc()
    REQUEST_LATENCY.labels(endpoint=endpoint).observe(process_time)

    return response

@app.get("/")
def home():
    return {"status": "ok", "message": "Weather Service Active"}

@app.get("/metrics")
def metrics():
    return Response(content=generate_latest(), media_type=CONTENT_TYPE_LATEST)

Writing app.py


In [5]:
%%writefile prometheus.yml
global:
  scrape_interval: 5s # Metrics fetch cycle

scrape_configs:
  - job_name: 'fastapi_python_app'
    metrics_path: '/metrics'
    static_configs:
      - targets: ['host.docker.internal:8000']

Writing prometheus.yml


In [ ]:
!uvicorn app:app --reload --port 8000

INFO:     Will watch for changes in these directories: ['/content']
INFO:     Uvicorn running on http://127.0.0.1:8000 (Press CTRL+C to quit)
INFO:     Started reloader process [4264] using WatchFiles
INFO:     Started server process [4266]
INFO:     Waiting for application startup.
INFO:     Application startup complete.
